# 📘 Notebook 1.1 — Database Schema, Table Roles & Data Contracts

## 1️⃣ Purpose of This Notebook

This notebook performs a **structural and semantic analysis** of the project database.

**Objective**:
- Establish a precise understanding of **what each table represents**.
- Define the **role of each table** (Base vs Derived).
- Set **constraints** for analytics and ML.

**Mandate**:
Machine learning correctness depends on semantics. This notebook formalizes the definitions that prevent leakage and ensure reproducibility.

---

## 2️⃣ Scope & Access Strategy

**Data Source**:
We analyze raw **CSV Exports** from `Dataset_Tables/csv_exports/`.

**Validation Goals**:
- **Census**: Row counts and file sizes.
- **Granularity**: Verifying intended primary keys.
- **Semantics**: Column definitions and roles.
- **Privacy**: Identifying PII.

---

In [ ]:
# Import required libraries
import pandas as pd
import os
import glob

# Define Path to CSV Data
DATA_DIR = '../../Dataset_Tables/csv_exports/'

def get_csv_files():
    return glob.glob(os.path.join(DATA_DIR, "*.csv"))

csv_files = get_csv_files()
print(f"Scanning directory: {os.path.abspath(DATA_DIR)}")
print(f"Found {len(csv_files)} CSV files.")
for f in csv_files:
    print(f" - {os.path.basename(f)}")

## 3️⃣ Table Census & Sizing

**Objective**: Understand the scale of data. 
- Small tables (< 100 rows) = Configuration/Lookup.
- Large tables (> 1M rows) = Fact/Event data.

In [ ]:
census_data = []

for file_path in csv_files:
    table_name = os.path.basename(file_path).replace('.csv', '')
    try:
        # Read first few lines to check structure, then full count if feasible
        df = pd.read_csv(file_path)
        row_count = df.shape[0]
        col_count = df.shape[1]
        cols = list(df.columns)
        census_data.append({'Table': table_name, 'Rows': row_count, 'Cols': col_count, 'Columns': str(cols)[:100] + '...'})
    except Exception as e:
        print(f"Error reading {table_name}: {e}")

census_df = pd.DataFrame(census_data).sort_values(by='Rows', ascending=False)
display(census_df)

## 4️⃣ Table Classification Framework

Each table is classified by **Origin** and **Analytical Role**.

### Origin Categories
- **Base Table**: Raw source data (Ground Truth).
- **Derived Table**: Computed metrics (Reproducible).
- **Output Table**: Results of ML processes.

### Analytical Roles
- **Label**: Observed outcome (Target).
- **Feature Source**: Input signals.
- **Metadata**: Context/Grouping.

## 5️⃣ Granularity & Primary Key Verification

**Objective**: Identify the logical primary key and verify uniqueness.

In [ ]:
def check_pk(file_pattern, pk_columns):
    files = glob.glob(os.path.join(DATA_DIR, file_pattern))
    if not files:
        print(f"File not found for pattern: {file_pattern}")
        return
        
    file_path = files[0]
    table_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    
    # Check if all PK columns exist
    missing_cols = [c for c in pk_columns if c not in df.columns]
    if missing_cols:
        print(f"Table: {table_name:<30} | Error: Columns not found {missing_cols}")
        return

    total_rows = len(df)
    unique_keys = df[pk_columns].drop_duplicates().shape[0]
    is_unique = (total_rows == unique_keys)
    
    print(f"Table: {table_name:<30} | PK: {str(pk_columns):<25} | Unique: {is_unique} ({unique_keys}/{total_rows})")

# Example checks - user should update these based on actual data
print("--- Primary Key Verification ---")
check_pk('location_metadata*.csv', ['location_id'])
check_pk('business_profile*.csv', ['business_id'])
check_pk('realtime_metrics*.csv', ['location_id', 'timestamp']) # Composite key example
check_pk('weather*.csv', ['weather_id'])

## 6️⃣ Data Sensitivity & Privacy Scan

**Objective**: Identify PII columns requiring redaction.

In [ ]:
sensitive_keywords = ['email', 'phone', 'address', 'password', 'name', 'ssn', 'ip_addr']
print("--- Potential PII Columns ---")

for file_path in csv_files:
    table_name = os.path.basename(file_path)
    df = pd.read_csv(file_path, nrows=0) # Only read headers
    for col in df.columns:
        if any(keyword in col.lower() for keyword in sensitive_keywords):
            print(f"Table: {table_name:<30} | Column: {col} [POTENTIAL SENSITIVE]")

## 7️⃣ Temporal & Relational Constraints

**Temporal Rules**:
- All time-series data must align on explicit timestamps.
- **NO FUTURE INFO**: Features cannot use future timestamps.

**Relational Rules**:
- `business_id` keys business analysis.
- `location_id` keys spatial context.
- Time-based joins must use (ID + Time).

---

## 8️⃣ Machine Learning Data Usage Rules

**The Contract**:
1. **Labels** (Daily_Revenue) must NEVER be used as Input Features.
2. **Derived Tables** must only depend on Base Tables.
3. **Output Tables** (Predictions) are strictly excluded from Training.

---

## 9️⃣ Final Artifacts

**Established Artifacts**:
- [x] Table Inventory (Census).
- [x] Classification (Roles defined).
- [x] Constraints (Temporal/Relational).
- [x] Binding Data Contract.

Proceed to **Notebook 1.2** for detailed profiling.